This notebook shows how to use GVL Predictor to predict the task progression of an episode.

First, make sure you select the Notebook kernel. Refer to Readme's Quickstart Guide section for creating the virtual environment. In Readme, we did not install the repo as a pip package, because we use `python -m `. Here, it is easier to run as a pip package, so:

In [1]:
!pip install -e ..

Looking in indexes: https://pypi.org/simple, https://pypi.ngc.nvidia.com
Obtaining file:///home/yihao/Downloads/software/gvl/wip/gvl-label-maker
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for gvl (pyproject.toml) ... done
  Created wheel for gvl: filename=gvl-0.1.0-0.editable-py3-none-any.whl size=6168 sha256=425eebe5652ef9915a8ff3aa4fa76ddf16d131c98e365a7501809158bc31b1a2
  Stored in directory: /tmp/pip-ephem-wheel-cache-_5w7z27v/wheels/c8/f6/9c/31c599adf0f5364a14aea530ad163f8e20e4c6e0a61a90cfe2
Successfully built gvl
  Attempting uninstall: gvl
    Found existing installation: gvl 0.1.0
    Uninstalling gvl-0.1.0:
      Successfully uninstalled gvl-0.1.0


Basic environment and output setup.

In [2]:
from __future__ import annotations
from pathlib import Path
import os
from dotenv import load_dotenv
from loguru import logger
from tqdm import tqdm

OUTPUT_DIR = Path("results/notebook-predict-episode-by-frame")
PROMPT_LOG_DIR = OUTPUT_DIR / "conversation_history"

load_dotenv(override=True)
logger.info("Environment variables loaded (dotenv)")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
os.environ["GVL_CONVERSATION_LOG_DIR"] = str(PROMPT_LOG_DIR)

2026-01-22 11:11:52.881 | INFO     | __main__:<module>:12 - Environment variables loaded (dotenv)


Instantiate data loader. The scripts in the `script` folder mostly use Hydra to configure the parameters. There configurations can be seen from `config` folder. However, here, to show what these parameters are, we use constants.

In [3]:
from gvl.data_loaders.mp4loader import MP4DataLoader

# Dataset
DATA_DIR = "results/test_data"  # Update this to your MP4 folder path
DATASET_NAME = "test_data"
INSTRUCTION_FILE = "instruction.txt"  # Or use instruction parameter directly
# INSTRUCTION = "Pick up the towel"  # Optional: set instruction directly
NUM_FRAMES = 15
NUM_CONTEXT_EPISODES = 3

# Data loader
SEED = 42
SHUFFLE = True
SAMPLING_METHOD = "random"
ANCHORING = ["first", "middle", "last"]

data_loader = MP4DataLoader(
    data_dir=DATA_DIR,
    instruction_file=INSTRUCTION_FILE,  # or use instruction="your task" instead
    num_frames=NUM_FRAMES,
    num_context_episodes=NUM_CONTEXT_EPISODES,
    shuffle=SHUFFLE,
    seed=SEED,
    sampling_method=SAMPLING_METHOD,
    anchoring=ANCHORING,
)

Load episode frames and anchors. We set the `ANCHORING` constant in the previous cell. 
- This tells the prediction agent what the important frames look like. 
  - In the original paper, the authors recommended using the first frame. 
  - In OpenGVL, there are options that you may choose first, middle, or last. 
- Here, we extend that to make it support a list. 
- Some of my preliminary experiments, using only the initial frame, showed that the performance may worsen as the task goes towards the end, which was the reason that I believe anchoring both the first and the last can help. However, this has not been valided systematically, other than some visual inspection confirming the second half of the prediction improved.

In [4]:
from datetime import datetime

EPISODE_INDEX = 0
ANCHOR_EPISODE_INDEX = 0

# Accepts anchoring as str (comma-separated) or list of str
def _normalize_anchoring(anchoring: str | list[str] | None) -> list[str]:
    if anchoring is None:
        return []
    if isinstance(anchoring, str):
        if "," in anchoring:
            return [part.strip() for part in anchoring.split(",") if part.strip()]
        return [anchoring]
    return [str(anchor) for anchor in anchoring]


def _select_anchor_frames(frames, anchoring: str | list[str] | None):
    if not frames:
        return [], []
    anchors: list = []
    anchor_kinds: list[str] = []
    choices = _normalize_anchoring(anchoring)
    if not choices:
        return anchors, anchor_kinds
    seen = set()
    for choice in choices:
        if choice == "first":
            anchor_idx = 0
            anchor_kind = "start"
        elif choice == "last":
            anchor_idx = len(frames) - 1
            anchor_kind = "last"
        elif choice == "middle":
            anchor_idx = len(frames) // 2
            anchor_kind = "middle"
        else:
            raise ValueError(f"Unknown anchoring method: {choice}")
        if anchor_idx in seen:
            continue
        anchors.append(frames[anchor_idx])
        anchor_kinds.append(anchor_kind)
        seen.add(anchor_idx)
    return anchors, anchor_kinds

frames, instruction = data_loader.load_episode_frames(episode_index=EPISODE_INDEX)
if not frames:
    raise ValueError(f"No frames loaded for episode_index={EPISODE_INDEX}")
logger.info(f"Loaded {len(frames)} frames for episode_index={EPISODE_INDEX} from {DATASET_NAME}")

anchor_frames_source, _ = data_loader.load_episode_frames(episode_index=ANCHOR_EPISODE_INDEX)
if not anchor_frames_source:
    raise ValueError(f"No frames loaded for anchor episode_index={ANCHOR_EPISODE_INDEX}")
anchor_frames, anchor_kinds = _select_anchor_frames(anchor_frames_source, ANCHORING)


2026-01-22 11:11:52.945 | INFO     | gvl.data_loaders.mp4loader:load_episode_frames:101 - Loading MP4 episode 0 from results/test_data/IMG_8426.mp4
2026-01-22 11:11:53.256 | INFO     | __main__:<module>:48 - Loaded 136 frames for episode_index=0 from test_data
2026-01-22 11:11:53.257 | INFO     | gvl.data_loaders.mp4loader:load_episode_frames:101 - Loading MP4 episode 0 from results/test_data/IMG_8426.mp4


We start with building a mapper. A mapper is an agent that parses some unstructured raw text outputs from the upstream agents. In this case, we need the progression score(s) of the frame(s). A raw output from the prediction agent usually contains sentences that are part of the agent's analysis, which is not needed. You may set the prediction agent's persona to avoid that, but here, we use a specialized agent, the mapper, to do that.

In [5]:
from dataclasses import dataclass
from gvl.mapper.gemini_mapper import GeminiMapper

# Mapper
MAPPER_MODEL_NAME = "gemini-2.5-flash-lite"
MAPPER_MAX_NEW_TOKENS = 1024
MAPPER_TEMPERATURE = 0.75
MAPPER_RETRIES = 3
MAPPING_PROMPT_TEMPLATE = """You are a specialized data extractor. Your sole purpose is to identify and extract task completion percentages for given frames from the provided text.

**Instructions:**

1.  **Identify Percentages:** Scan the user's message and identify all occurrences of task completion percentages. These are typically expressed as numbers followed by a percent sign (e.g., "5%", "30%", etc.). Don't take into consideration overall task completion, starting frame completion, or any other non-specific percentages. If frame i appears multiple times, extract only one percentage for that frame (the first one).
2.  **Extract Numerical Values:** From each identified percentage, extract only the numerical value. For example, from "5%", you will extract the number `5`.
3.  **Format as JSON:** Compile all extracted numerical values into a single JSON object. The JSON object must be in the following format: `{"prediction": [list_of_percentages]}`.
4.  **No Percentages Found:** If the user's message contains no percentage values, return an empty list within the JSON object, like this: `{"prediction": []}`.

**Important:** 
- Your response must **only** contain the final JSON object. Do not include any additional text, explanations, apologies, or markdown formatting. 
- Don't take into consideration overall task completion, starting frame completion, or any other non-specific percentages.
- Make sure the number of frames in the list corresponds to the number of unique frames mentioned in the user's message. Do not add extra numbers or omit any frames.

---

### **Example:**

**User Message:**
`As an expert roboticist, I will now analyze each frame to predict the task completion percentage for the task of "open door". The analysis for each frame is independent and based on a decomposition of the task into key stages: approaching the handle, grasping the handle, and pulling the door open.

**Initial State (0% completion):** The robot is positioned before the closed cabinet, ready to begin the task.
**Final State (100% completion):** Frame 1: The robot is approaching the cup. Task Completion: 5% Frame 2: The robot is grasping the cup. Task Completion: 30% Frame 3: The robot is lifting the cup. Task Completion: 60% Frame 4: The robot is pouring the liquid. Task Completion: 80% Frame 5: The liquid is flowing into the cup. Task Completion: 90% Frame 6: The pouring is almost complete. Task Completion: 95% Frame 7: The robot is retracting its arm. Task Completion: 98% Frame 8: The robot is returning to its initial position. Task Completion: 99% Frame 9: The robot is back to its initial position. Task Completion: 99% Frame 10: The robot is back to its initial position. Task Completion: 99% Frame 11: The robot is back to its initial position. Task Completion: 99% Frame 12: The robot is back to its initial position. Task Completion: 99% Frame 13: The robot is back to its initial position. Task Completion: 99% Frame 14: The robot is back to its initial position. Task Completion: 99% Frame 15: The robot is back to its initial position. Task Completion: 99% Frame 16: The robot is back to its initial position. Task Completion: 99% Frame 17: The robot is back to its initial position. Task Completion: 99% Frame 18: The robot is back to its initial position. Task Completion: 99% Frame 19: The robot is back to its initial position. Task Completion: 99% Frame 20: The robot is back to its initial position. Task Completion: 100%. Final Task Completion Percentage: 88.0%`

**Your Response:**
```json
{"prediction": [5, 30, 60, 80, 90, 95, 98, 99, 99, 99, 99, 99, 99, 99, 99, 99, 99, 99, 99, 100]}
```

Answer:
"""

@dataclass
class MappingPrompt:
    template: str
    name: str = "default"

mapping_prompt = MappingPrompt(template=MAPPING_PROMPT_TEMPLATE)
mapper = GeminiMapper(
    model_name=MAPPER_MODEL_NAME,
    max_new_tokens=MAPPER_MAX_NEW_TOKENS,
    temperature=MAPPER_TEMPERATURE,
    retries=MAPPER_RETRIES,
    mapping_prompt=mapping_prompt,
)

Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


Then, we build the prediction agent's client.

In [6]:
# Model
from gvl.clients.qwen3 import Qwen3Client

MODEL_CLASS = Qwen3Client
MODEL_NAME = "Qwen/Qwen3-VL-32B-Instruct"
MODEL_MAX_INPUT_LENGTH = 32000
client = MODEL_CLASS(model_name=MODEL_NAME, max_input_length=MODEL_MAX_INPUT_LENGTH)
logger.info("predict_episode_by_frame uses the full episode; sampling is disabled for eval frames.")
model_name_safe = client.model_name.replace("/", "_")
starting_time = datetime.now().isoformat().replace(":", "-")

PROMPT_TEMPLATE = """You are an expert roboticist tasked to predict task completion percentages for frames of a robot for the task of {instruction}.
The task completion percentages are between 0 and 100, where 100 corresponds to full task completion.
The frames may be in random order; reason about each frame independently when estimating completion.
Make sure the number of frames in the list corresponds to the number of unique frames mentioned in the user's message in your final answer.
There are {num_frames} frames. Do not add extra numbers or omit any frames.
"""

PROMPT_PHRASES = {
    "anchor_scene_label_start": "Initial robot scene:",
    "anchor_scene_completion_start": "In the initial robot scene, the task completion percentage is 0%.",
    "anchor_scene_label_middle": "Middle robot scene:",
    "anchor_scene_completion_middle": "In the middle robot scene, the task completion percentage is 50%.",
    "anchor_scene_label_last": "Last robot scene:",
    "anchor_scene_completion_last": "In the last robot scene, the task completion percentage is 100%.",
    "context_frame_label_template": "Frame {i}:",
    "context_frame_completion_template": "Task Completion Percentage: {p}%",
    "eval_frame_label_template": "Frame {i}:",
    "eval_task_completion_instruction": [
        "Now, for the task of {instruction}, output the task completion percentage for the following frames that are presented in random order. For each frame, format your response as follow: Frame {{i}}: Task Completion Percentages:{{}}%",
        "Be rigorous and precise; percentage reflects task completion. There are {num_frames} frames. Do not add extra numbers or omit any frames.",
        "Remember: frames are in random order.",
    ],
}

/home/yihao/miniconda3/envs/gvl-test2/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Skipping import of cpp extensions due to incompatible torch version 2.7.1+cu128 for torchao version 0.14.1             Please see https://github.com/pytorch/ao/issues/2919 for more info
2026-01-22 11:11:57.534 | INFO     | gvl.clients.qwen3:__init__:23 - Loading Qwen3 model Qwen/Qwen3-VL-32B-Instruct ...
`torch_dtype` is deprecated! Use `dtype` instead!
Loading checkpoint shards: 100%|██████████| 14/14 [00:10<00:00,  1.33it/s]
2026-01-22 11:12:10.506 | INFO     | gvl.clients.qwen3:__init__:30 - <class 'transformers.models.qwen3_vl.processing_qwen3_vl.Qwen3VLProcessor'>
2026-01-22 11:12:10.507 | INFO     | __main__:<module>:8 - predict_episode_by_frame uses the full episode; sampling is disabled for eval frames.


We load context episodes. These are the episodes frames that serve as the examples. Refer to the functions in this cell to check how the shuffling is done and how a `ContextEpisodes` class is instantiated.

In [7]:
from gvl.utils.data_types import ContextEpisodes

CONTEXT_EPISODE_INDICES = [2, 3, 4]

if CONTEXT_EPISODE_INDICES:
    logger.info(f"Using explicit context_episode_indices={CONTEXT_EPISODE_INDICES}")
    contexts = []
    original_num_frames = data_loader.num_frames
    try:
        for ctx_idx in CONTEXT_EPISODE_INDICES:
            ctx_frames, ctx_instruction = data_loader.load_episode_frames(episode_index=int(ctx_idx))
            if not ctx_frames:
                logger.warning(f"No frames loaded for context episode_index={ctx_idx}; skipping")
                continue
            contexts.append(
                data_loader._build_episode(
                    frames=ctx_frames,
                    instruction=ctx_instruction,
                    episode_index=int(ctx_idx),
                    sampling_method=data_loader.sampling_method,
                    anchoring=data_loader.anchoring,
                )
            )
    finally:
        data_loader.num_frames = original_num_frames
    context_episodes = ContextEpisodes(contexts)
else:
    context_episodes = ContextEpisodes([])

2026-01-22 11:12:10.512 | INFO     | __main__:<module>:6 - Using explicit context_episode_indices=[2, 3, 4]
2026-01-22 11:12:10.512 | INFO     | gvl.data_loaders.mp4loader:load_episode_frames:101 - Loading MP4 episode 2 from results/test_data/IMG_8428.mp4
2026-01-22 11:12:10.877 | INFO     | gvl.data_loaders.mp4loader:load_episode_frames:101 - Loading MP4 episode 3 from results/test_data/IMG_8429.mp4
2026-01-22 11:12:11.218 | INFO     | gvl.data_loaders.mp4loader:load_episode_frames:101 - Loading MP4 episode 4 from results/test_data/IMG_8430.mp4


Prepare for the prediction.

In [8]:
FRAME_STRIDE = 10  # 0 = every frame, 1 = every other frame
SAVE_RAW = True
TEMPERATURE = 1.0

import json
import time

from gvl.metrics.frame_error import FrameProgressErrorMetric
from gvl.utils import inference as infer_utils
from gvl.utils.data_types import EvalFrame, FrameEvalCase

# Ground truth
def _compute_task_completion_rate(frame_index: int, total_frames: int) -> int | None:
    if total_frames <= 0:
        return None
    if total_frames == 1:
        return 100
    return round(frame_index / (total_frames - 1) * 100)


frame_metric = FrameProgressErrorMetric()
frame_records = []
predicted_values: list[int | None] = [None] * len(frames)

frame_jsonl_path = OUTPUT_DIR / f"{model_name_safe}_{starting_time}_episode_{EPISODE_INDEX}_frame_predictions.jsonl"
logger.info(f"Streaming per-frame predictions to {frame_jsonl_path}")

if FRAME_STRIDE < 0:
    raise ValueError("FRAME_STRIDE must be >= 0")

total_frames = len(frames)
frame_step = FRAME_STRIDE + 1
selected_positions = list(range(0, total_frames, frame_step))
logger.info(
    f"Predicting {len(selected_positions)}/{total_frames} frames in original order "
    f"with frame_stride={FRAME_STRIDE}"
)

2026-01-22 11:12:11.746 | INFO     | __main__:<module>:26 - Streaming per-frame predictions to results/notebook-predict-episode-by-frame/Qwen_Qwen3-VL-32B-Instruct_2026-01-22T11-12-10.507667_episode_0_frame_predictions.jsonl
2026-01-22 11:12:11.746 | INFO     | __main__:<module>:34 - Predicting 13/136 frames in original order with frame_stride=10


The prediction loop.

In [9]:

with frame_jsonl_path.open("w", encoding="utf-8") as f:
    timing_path = OUTPUT_DIR / f"{model_name_safe}_{starting_time}_episode_{EPISODE_INDEX}_frame_timings.txt"
    timing_file = timing_path.open("w", encoding="utf-8")
    for loop_idx, pos in enumerate(selected_positions):
        frame = frames[pos]
        gt_rate = _compute_task_completion_rate(pos, total_frames)
        eval_frame = EvalFrame(
            instruction=instruction,
            frame=frame,
            anchor_frames=anchor_frames,
            anchor_kinds=anchor_kinds,
            task_completion_rate=gt_rate,
        )
        frame_eval_case = FrameEvalCase(eval_frame=eval_frame, context_episodes=context_episodes)
        t0 = time.time()
        record = infer_utils.predict_on_frame_eval_case(
            idx=loop_idx,
            total=len(selected_positions),
            eval_case=frame_eval_case,
            client=client,
            prompt_template=PROMPT_TEMPLATE,
            save_raw=SAVE_RAW,
            frame_metric=frame_metric,
            dataset_name=DATASET_NAME,
            temperature=float(TEMPERATURE),
            mapper=mapper,
            prompt_phrases=PROMPT_PHRASES,
        )
        duration = time.time() - t0
        frame_records.append(record)
        predicted_values[pos] = record.predicted_percentage
        f.write(json.dumps(record.to_dict(include_images=False), ensure_ascii=False) + "\n")
        f.flush()
        timing_file.write(f"{loop_idx}\tframe_idx={pos}\tduration_sec={duration:.3f}\n")
        timing_file.flush()
    timing_file.close()


2026-01-22 11:12:11.752 | INFO     | gvl.utils.inference:predict_on_frame_eval_case:227 - Processing frame eval case 1/13 from test_data
2026-01-22 11:12:11.752 | DEBUG    | gvl.utils.inference:_generate_eval_case_response:95 - Prompt (truncated to 400 chars): You are an expert roboticist tasked to predict task completion percentages for frames of a robot for the task of Move the Apple Pencil from the left of a Logitech Mouse to the right of the Logitech Mouse..
The task completion percentages are between 0 and 100, where 100 corresponds to full task completion.
The frames may be in random order; reason about each frame independently when estimating com...
2026-01-22 11:12:11.752 | DEBUG    | gvl.clients.base:_generate_with_retry:63 - Model generation attempt 1/4
2026-01-22 11:12:12.918 | INFO     | gvl.clients.qwen3:_generate_from_events:57 - Input length: 4100
2026-01-22 11:12:15.328 | INFO     | gvl.clients.base:_generate_with_retry:66 - Model response length: 40 characters
2026-01-

Log the metrics.

In [10]:
from gvl.metrics.voc import value_order_correlation


def _aggregate_error_counts(records) -> dict[str, int]:
    totals: dict[str, int] = {}
    for record in records:
        for key, value in record.error_count.items():
            totals[key] = totals.get(key, 0) + value
    return totals


missing_predictions = sum(value is None for value in predicted_values)
if missing_predictions:
    logger.warning(f"{missing_predictions} frame predictions missing; leaving as None.")

error_count_total = _aggregate_error_counts(frame_records)
if missing_predictions:
    error_count_total["MissingPrediction"] = missing_predictions

available_idx = [i for i, v in enumerate(predicted_values) if v is not None]
if len(available_idx) < 2:
    metrics_payload = {
        "voc": 0.0,
        "voc_note": "insufficient predictions",
    }
else:
    preds = [predicted_values[i] for i in available_idx]  # type: ignore[index]
    if len(frames) > 1:
        truth = [round(i / (len(frames) - 1) * 100) for i in available_idx]
    else:
        truth = [100 for _ in available_idx]
    voc_value = value_order_correlation(preds, truth)
    if voc_value != voc_value:  # NaN
        metrics_payload = {"voc": 0.0, "voc_note": "undefined correlation"}
    else:
        metrics_payload = {"voc": float(voc_value)}
        if missing_predictions or sum(error_count_total.values()) > 0:
            metrics_payload["voc_note"] = "computed on predicted frames only"

    print(f"Predictions: {preds}")
    print(f"Ground truth: {truth}")
    print(f"VOC: {voc_value}")

episode_record = {
    "index": 0,
    "dataset": DATASET_NAME,
    "episode_index": EPISODE_INDEX,
    "predicted_percentages": predicted_values,
    "valid_length": missing_predictions == 0,
    "metrics": metrics_payload,
    "error_count": error_count_total,
}

logger.info(f"Wrote per-frame predictions to {frame_jsonl_path}")

episode_json_path = OUTPUT_DIR / f"{model_name_safe}_{starting_time}_episode_{EPISODE_INDEX}_prediction.json"
with episode_json_path.open("w", encoding="utf-8") as f:
    json.dump(episode_record, f, indent=2)
logger.info(f"Wrote episode prediction summary to {episode_json_path}")

2026-01-22 11:12:59.302 | WARNING  | __main__:<module>:14 - 123 frame predictions missing; leaving as None.
2026-01-22 11:12:59.303 | INFO     | __main__:<module>:54 - Wrote per-frame predictions to results/notebook-predict-episode-by-frame/Qwen_Qwen3-VL-32B-Instruct_2026-01-22T11-12-10.507667_episode_0_frame_predictions.jsonl
2026-01-22 11:12:59.304 | INFO     | __main__:<module>:59 - Wrote episode prediction summary to results/notebook-predict-episode-by-frame/Qwen_Qwen3-VL-32B-Instruct_2026-01-22T11-12-10.507667_episode_0_prediction.json


Predictions: [0, 0, 31, 42, 45, 50, 50, 50, 88, 97, 98, 100, 98]
Ground truth: [0, 8, 16, 24, 33, 41, 49, 57, 65, 73, 81, 90, 98]
VOC: 0.9834134593578462


Save the predicted video with a progression curve overlay.

In [11]:
from PIL import ImageDraw

from gvl.utils.frame import save_progress_video
from gvl.utils.images import to_pil

VIDEO_FPS = 50


def _save_progress_curve_video(
    frames,
    progress_values,
    output_path: Path,
    *,
    fps: int = 2,
    plot_width: int = 220,
    plot_height: int = 80,
) -> Path:
    """Save a video with a small progress curve overlay on each frame."""
    try:
        import numpy as np
        import imageio.v2 as imageio
    except ImportError as exc:
        raise RuntimeError("imageio is required to save MP4 files; install imageio and imageio-ffmpeg") from exc

    output_path.parent.mkdir(parents=True, exist_ok=True)
    if not frames:
        raise ValueError("No frames provided for video output.")

    with imageio.get_writer(output_path, fps=fps, codec="libx264") as writer:
        for idx, frame in enumerate(frames):
            pil = to_pil(frame).convert("RGB")
            draw = ImageDraw.Draw(pil)
            # Plot area in bottom-left
            margin = 10
            left = margin
            bottom = pil.height - margin
            top = bottom - plot_height
            right = left + plot_width
            # Semi-transparent grey background patch for the plot
            patch = ImageDraw.ImageDraw(pil, "RGBA")
            patch.rectangle([left, top, right, bottom], fill=(128, 128, 128, 180))
            draw.rectangle([left, top, right, bottom], outline=(255, 255, 255))
            # Build curve up to current frame using only available predictions
            points = []
            seen = [j for j, v in enumerate(progress_values[: idx + 1]) if v is not None]
            if len(seen) >= 2:
                denom = max(1, len(progress_values) - 1)
                for j in seen:
                    v = progress_values[j]
                    x = left + int(plot_width * j / denom)
                    v_clamped = max(0, min(100, int(v)))  # type: ignore[arg-type]
                    y = bottom - int(plot_height * v_clamped / 100)
                    points.append((x, y))
                if len(points) > 1:
                    draw.line(points, fill=(255, 255, 255), width=2)
            writer.append_data(np.array(pil))
    return output_path


video_fps = int(VIDEO_FPS)
display_values = []
last_pred = None
for v in predicted_values:
    if v is not None:
        last_pred = v
    display_values.append(last_pred)

video_path = OUTPUT_DIR / f"{model_name_safe}_{starting_time}_episode_{EPISODE_INDEX}_pred.mp4"
logger.info(f"Saving prediction video in original order at {video_path}")
try:
    save_progress_video(
        frames,
        display_values,
        video_path,
        label_prefix="pred",
        fps=video_fps,
    )
except Exception as exc:
    logger.exception(f"Failed to save prediction video at {video_path}: {exc}")
    logger.error("MP4 saving requires imageio + imageio-ffmpeg (and ffmpeg).")
else:
    logger.info(f"Wrote prediction video to {video_path}")

curve_video_path = OUTPUT_DIR / f"{model_name_safe}_{starting_time}_episode_{EPISODE_INDEX}_pred_curve.mp4"
logger.info(f"Saving prediction video with progress curve at {curve_video_path}")
try:
    _save_progress_curve_video(
        frames,
        predicted_values,
        curve_video_path,
        fps=video_fps,
    )
except Exception as exc:
    logger.exception(f"Failed to save prediction curve video at {curve_video_path}: {exc}")
    logger.error("MP4 saving requires imageio + imageio-ffmpeg (and ffmpeg).")
else:
    logger.info(f"Wrote prediction curve video to {curve_video_path}")

2026-01-22 11:12:59.322 | INFO     | __main__:<module>:69 - Saving prediction video in original order at results/notebook-predict-episode-by-frame/Qwen_Qwen3-VL-32B-Instruct_2026-01-22T11-12-10.507667_episode_0_pred.mp4
IMAGEIO FFMPEG_WRITER WARNING: input image is not divisible by macro_block_size=16, resizing from (244, 244) to (256, 256) to ensure video compatibility with most codecs and players. To prevent resizing, make your input image divisible by the macro_block_size or set the macro_block_size to 1 (risking incompatibility).
2026-01-22 11:12:59.989 | INFO     | __main__:<module>:82 - Wrote prediction video to results/notebook-predict-episode-by-frame/Qwen_Qwen3-VL-32B-Instruct_2026-01-22T11-12-10.507667_episode_0_pred.mp4
2026-01-22 11:12:59.990 | INFO     | __main__:<module>:85 - Saving prediction video with progress curve at results/notebook-predict-episode-by-frame/Qwen_Qwen3-VL-32B-Instruct_2026-01-22T11-12-10.507667_episode_0_pred_curve.mp4
IMAGEIO FFMPEG_WRITER WARNING: 

Cleanups.

In [12]:
from gvl.utils.cleanup import cleanup_resources

# Uncomment the following line to cleanup resources after prediction
cleanup_resources(clients=[client, mapper], records=frame_records)

2026-01-22 11:13:00.601 | DEBUG    | gvl.utils.cleanup:_cleanup_client:44 - Called close on Qwen3Client
2026-01-22 11:13:00.602 | DEBUG    | gvl.utils.cleanup:_cleanup_client:51 - Cleared Qwen3Client.model
2026-01-22 11:13:00.602 | DEBUG    | gvl.utils.cleanup:_cleanup_client:51 - Cleared Qwen3Client.processor
2026-01-22 11:13:00.603 | DEBUG    | gvl.mapper.gemini_mapper:close:145 - Closed Gemini mapper client
2026-01-22 11:13:00.603 | DEBUG    | gvl.utils.cleanup:_cleanup_client:44 - Called close on GeminiMapper
2026-01-22 11:13:00.603 | DEBUG    | gvl.utils.cleanup:_cleanup_client:51 - Cleared GeminiMapper.client
2026-01-22 11:13:00.604 | DEBUG    | gvl.utils.cleanup:_clear_container:31 - Cleared records
2026-01-22 11:13:00.652 | DEBUG    | gvl.utils.cleanup:cleanup_resources:73 - Cleared torch CUDA cache
2026-01-22 11:13:01.037 | DEBUG    | gvl.utils.cleanup:cleanup_resources:77 - Garbage collection complete
